# Stage-II hyperparameter sweep

Searches the training recipe for MELD on **dev macro-F1**. RoBERTa-base is fixed by the project specification, so only the recipe varies.

In [ ]:
import os, shutil, subprocess, sys, time, zipfile, pathlib

WORK = "/kaggle/working"
EXPECTED_SOURCE = ""  # set by kaggle_run.py submit
STAGES = ["stage1", "M1", "M2", "M3", "M4"]   # edit to run a subset
EXTRA = []                                     # e.g. ["--set", "epochs=3"]

INPUT_ROOT = pathlib.Path("/kaggle/input")
NEEDED = {"src", "scripts", "configs"}
ZIPPED = {f"{name}.zip" for name in NEEDED}


def find_source_root(root: pathlib.Path = INPUT_ROOT, max_depth: int = 6) -> pathlib.Path:
    """Locate the attached dataset directory holding src/ scripts/ configs/.

    Kaggle mounts datasets at different depths (historically /kaggle/input/<slug>,
    now /kaggle/input/datasets/<owner>/<slug>/...), so search rather than assume.
    """
    if not root.exists():
        raise FileNotFoundError("/kaggle/input does not exist -- no inputs attached")
    for current, directories, files in os.walk(root):
        depth = len(pathlib.Path(current).relative_to(root).parts)
        if depth > max_depth:
            directories[:] = []
            continue
        names = set(directories) | set(files)
        if NEEDED <= names or ZIPPED <= names:
            return pathlib.Path(current)
    listing = "\n".join(f"  {p}" for p in sorted(root.rglob("*"))[:40]) or "  (empty)"
    raise FileNotFoundError(
        "no attached dataset contains src/ scripts/ configs/. Attach the private dataset "
        f"f2c-src (Add Input > Datasets > Your Datasets) and re-run. Saw:\n{listing}"
    )


SRC = find_source_root()
print("source root:", SRC)

for name in ("src", "scripts", "configs"):
    target = pathlib.Path(WORK, name)
    if target.exists():
        shutil.rmtree(target)
    source_dir = SRC / name
    archive = SRC / f"{name}.zip"
    if source_dir.is_dir():
        shutil.copytree(source_dir, target)
    else:
        with zipfile.ZipFile(archive) as bundle:
            bundle.extractall(WORK if bundle.namelist()[0].startswith(f"{name}/") else target)

os.chdir(WORK)
sys.path.insert(0, f"{WORK}/src")
print("staged:", sorted(p.name for p in pathlib.Path(WORK).iterdir()))

# Stop before training if Kaggle mounted an older upload of the source dataset.
mounted = pathlib.Path(WORK, "src", "SOURCE_ID")
mounted = mounted.read_text().strip() if mounted.exists() else "(none)"
print("source fingerprint: mounted", mounted, "| expected", EXPECTED_SOURCE or "(unchecked)")
if EXPECTED_SOURCE and mounted != EXPECTED_SOURCE:
    raise SystemExit(
        f"stale source: Kaggle mounted {mounted}, this notebook expects {EXPECTED_SOURCE}. "
        "The new dataset version has not propagated yet; wait a few minutes and resubmit."
    )

# nvidia-smi is absent from some Kaggle images even when a GPU is attached, so it is
# only a hint here; torch decides in the next cell.
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                         capture_output=True, text=True).stdout.strip() or "nvidia-smi: no output")
except FileNotFoundError:
    print("nvidia-smi not installed in this image")

In [ ]:
!pip install -q "transformers==5.0.0" "datasets>=2.16" "scikit-learn>=1.4" "pyyaml>=6.0" 2>&1 | tail -2

ALLOW_CPU = False   # CPU training will not finish inside a 12-hour session

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif ALLOW_CPU:
    print("WARNING: no GPU, running on CPU")
else:
    raise SystemExit(
        "No GPU attached. In the Kaggle notebook editor set Settings > Accelerator to "
        "GPU T4 x2 (or P100) and check your weekly GPU quota, then re-run. "
        "Set ALLOW_CPU = True to proceed anyway."
    )

In [ ]:
def run(cmd):
    """Stream a pipeline step and fail loudly so the kernel stops on error."""
    print("\n>>>", " ".join(cmd), flush=True)
    started = time.time()
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        if "it/s]" in line or "s/it]" in line:      # collapse tqdm spam
            continue
        print(line, end="", flush=True)
    code_ = process.wait()
    print(f"<<< exit={code_} after {(time.time() - started) / 60:.1f} min", flush=True)
    if code_ != 0:
        raise RuntimeError(f"step failed: {' '.join(cmd)}")

## Stage I backbones

In [ ]:
run([sys.executable, "scripts/prepare_data.py", "--dataset", "all"])

SEED = 42
BACKBONE = "checkpoints/stage1_sweep"

# Two Stage-I backbones: the default single-label reduction, and the one that keeps
# every example by taking its first label (5,767 more training examples).
run([sys.executable, "scripts/train_stage1.py", "--config", "configs/stage1.yaml",
     "--set", "run_name=stage1_sweep", f"seed={SEED}"])
run([sys.executable, "scripts/train_stage1.py", "--config", "configs/stage1.yaml",
     "--set", "run_name=stage1_sweep_first", f"seed={SEED}", "multilabel_policy=first"])

## Stage-II grid (system M4)

In [ ]:
# Each entry is (name, extra --set overrides). Selection is on DEV macro-F1 only;
# the test numbers these runs also produce are not used to choose anything.
GRID = [
    ("base",                 []),
    ("focal2",               ["loss=focal", "focal_gamma=2.0"]),
    ("focal1",               ["loss=focal", "focal_gamma=1.0"]),
    ("weighted_ce",          ["class_weighted_loss=true"]),
    ("focal2_weighted",      ["loss=focal", "focal_gamma=2.0", "class_weighted_loss=true"]),
    ("smooth",               ["label_smoothing=0.1"]),
    ("long",                 ["epochs=12"]),
    ("long_focal",           ["epochs=12", "loss=focal", "focal_gamma=2.0"]),
    ("long_weighted",        ["epochs=12", "class_weighted_loss=true"]),
    ("head_lr3e4",           ["head_learning_rate=3.0e-4"]),
    ("backbone_lr2e5",       ["learning_rate=2.0e-5"]),
    ("ute4",                 ["ute_layers=4"]),
    ("speaker",              ["speaker_aware=true"]),
    ("long_focal_speaker",   ["epochs=12", "loss=focal", "focal_gamma=2.0", "speaker_aware=true"]),
    ("speaker_prefix",       ["speaker_prefix=true"]),
    ("long_focal_prefix",    ["epochs=12", "loss=focal", "focal_gamma=2.0", "speaker_prefix=true"]),
    ("stage1_first",         [], "checkpoints/stage1_sweep_first"),
]

failed = []
for entry in GRID:
    name, overrides = entry[0], entry[1]
    backbone = entry[2] if len(entry) > 2 else BACKBONE
    try:
        run([sys.executable, "scripts/train_stage2.py", "--system", "M4", "--set",
             f"run_name=sweep_{name}", f"seed={SEED}", f"from_stage1={backbone}", *overrides])
    except RuntimeError as error:      # one bad config must not sink the whole grid
        print("FAILED:", name, error, flush=True)
        failed.append(name)
print("\nfailed configs:", failed or "none")

## Ranking

In [ ]:
import json, pathlib

rows = []
for path in sorted(pathlib.Path("results").glob("sweep_*/metrics.json")):
    payload = json.loads(path.read_text())
    rows.append((payload["best_dev_macro_f1"], path.parent.name, payload["test"]["macro_f1"]))

rows.sort(reverse=True)
print(f"{'run':<28}{'DEV macro-F1':>14}{'(test)':>10}")
for dev, name, test in rows:
    print(f"{name:<28}{dev:>14.4f}{test:>10.4f}")
print("\nselection is by the DEV column; test is shown only for the record")

## Trim output

In [ ]:
import shutil, pathlib
for junk in ("src", "scripts", "configs", "data", "checkpoints", "__pycache__"):
    shutil.rmtree(pathlib.Path("/kaggle/working", junk), ignore_errors=True)
print(sorted(p.name for p in pathlib.Path("/kaggle/working").iterdir()))